# Stacking on fold B (second-level LightGBM over matcher features + p_lgbm + p_ce)

- CPU session is enough (GPU also fine). **Internet ON** only for pip.
- Inputs: the challenge data, the code dataset (version with `src/stack.py`), the **run-4 output of
  `run_kaggle.ipynb`** (its `cache/`) and the **output of `run_kaggle_ce.ipynb`** (its `cache_ce/`). Both are read in
  place (read-only).
- Writes `/kaggle/working/output_stack/` (`matching_results.tsv`, `candidate_pairs.tsv` = run 4's candidates,
  `variants/`, `stack.json`, `test_pair_probs_stack.parquet`), fold models in `/kaggle/working/stack_models/`
  and the log `/kaggle/working/stack.log`.

## Step 1: locate the inputs and set paths
Finds the challenge data, the code dataset (copied to `/kaggle/working`), the run-4 `cache/` (features, matcher
OOF, pre-filter keep set, test p_lgbm) and the cross-encoder caches (`cache_ce/`, and `cache_ce2/` / `cache_ce3/`
where used). All inputs are read in place; missing or outdated inputs stop the notebook with a clear message.

In [ ]:
import glob, os, shutil, subprocess, sys, hashlib

found_data = sorted(glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True))
assert found_data, "challenge data not found"
DATA_DIR = os.path.dirname(os.path.dirname(found_data[0]))
WORK = "/kaggle/working"
WORK_CODE = f"{WORK}/business_entity_resolution"
hits = sorted(glob.glob("/kaggle/input/**/business_entity_resolution/src/stack.py", recursive=True))
assert hits, "code dataset with src/stack.py not found (attach the new amlc26-code version)"
shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), WORK_CODE, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("__pycache__"))
caches = [os.path.dirname(os.path.dirname(p)) for p in
          sorted(glob.glob("/kaggle/input/**/cache/prefilter/test_keep.parquet", recursive=True))]
caches = [c for c in caches if os.path.exists(f"{c}/predict/test_probs_pf.parquet")
          and os.path.exists(f"{c}/features/test.parquet")]
assert caches, "run-4 output (cache/ with prefilter, predict and features) not attached"
RUN4_CACHE = caches[0]
RUN4_CAND = os.path.join(os.path.dirname(RUN4_CACHE), "output", "candidate_pairs.tsv")
ce_hits = sorted(glob.glob("/kaggle/input/**/cache_ce/ce_test.parquet", recursive=True))
assert ce_hits, "output of run_kaggle_ce.ipynb (cache_ce/ce_test.parquet) not attached"
CE_DIR = os.path.dirname(ce_hits[0])
OUT, MODELS, LOG = f"{WORK}/output_stack", f"{WORK}/stack_models", f"{WORK}/stack.log"
validators = sorted(glob.glob("/kaggle/input/**/utils/validate_submission.py", recursive=True))
os.environ.update(BER_DATA_DIR=DATA_DIR, BER_CACHE_DIR=RUN4_CACHE, PYTHONIOENCODING="utf-8")
for k, v in dict(data=DATA_DIR, run4_cache=RUN4_CACHE, run4_candidates=RUN4_CAND, cache_ce=CE_DIR,
                 validator=validators[0] if validators else "NOT FOUND").items():
    print(f"{k:16s} {v}")
print("cache_ce files:", sorted(os.listdir(CE_DIR)))

## Step 2: install packages
Small pure-Python packages, then a clean polars 1.44.2 + lightgbm 4.7.0 install and an import check in a
subprocess (stops the notebook if the environment is broken).

In [ ]:
# small pure packages; polars reinstalled cleanly (the image may ship another build without its runtime)
!pip install -q --no-deps rapidfuzz==3.14.6 anyascii==0.3.3 Metaphone==0.6 2>&1 | tail -2
!pip uninstall -y -q polars polars-lts-cpu polars-u64-idx polars-runtime-32 polars-runtime-64 polars-runtime-compat 2>&1 | tail -2
!pip install -q polars==1.44.2 lightgbm==4.7.0 2>&1 | tail -2
chk = subprocess.run([sys.executable, "-c", "import polars as pl, lightgbm; print('polars', pl.__version__, 'lightgbm', "
                      "lightgbm.__version__); print(pl.DataFrame({'a': [1, 2]}).lazy().select(pl.col('a').sum()).collect().item())"],
                     capture_output=True, text=True)
print(chk.stdout, chk.stderr[-2000:])
assert chk.returncode == 0 and "polars 1.44.2" in chk.stdout, "polars 1.44.2 is not importable: fix before running"
!nproc && free -g | head -2

## Step 3: run the stacking
Builds fold B (the cross-encoders' held-out S1s, p_lgbm out-of-fold), trains the second-level LightGBM variants
with 5-fold GroupKFold by S1, prints the fold-B table and the verdict, then scores all test candidates and writes
the submission files. Output is also appended to the log file; a non-zero exit code raises.

In [ ]:
cmd = (f"set -o pipefail; {sys.executable} -u {WORK_CODE}/src/stack.py --ce_dir {CE_DIR} --stack_dir {MODELS} "
       f"--out {OUT} 2>&1 | tee -a {LOG}")
print(">>>", cmd, flush=True)
rc = subprocess.run(cmd, shell=True, executable="/bin/bash").returncode
print("exit code:", rc)
if rc != 0:
    raise RuntimeError(f"stack.py failed with exit code {rc}: see the output above / {LOG}")

## Step 4: check and validate the outputs
`candidate_pairs.tsv` must be byte-identical to run 4's (the stacking only re-scores those pairs; it is replaced by
run 4's file otherwise), and every written `matching_results*.tsv` is checked with `utils/validate_submission.py`.

In [ ]:
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()
if os.path.exists(RUN4_CAND):
    same = md5(RUN4_CAND) == md5(f"{OUT}/candidate_pairs.tsv")
    print("output_stack/candidate_pairs.tsv identical to run 4's:", same)
    if not same:
        shutil.copy2(RUN4_CAND, f"{OUT}/candidate_pairs.tsv")
        print("-> replaced by run 4's file (the stack only uses those pairs)")
if validators:
    for f in [f"{OUT}/matching_results.tsv"] + sorted(glob.glob(f"{OUT}/variants/*.tsv")):
        out = subprocess.run([sys.executable, validators[0], "--matching", f, "--candidate", f"{OUT}/candidate_pairs.tsv",
                              "--test-dir", f"{DATA_DIR}/test"], capture_output=True, text=True)
        verdict = [l for l in out.stdout.splitlines() if l.startswith(("PASS", "FAIL"))]
        print(os.path.basename(f), verdict)
        open(LOG, "a").write(f"validator {f}: {verdict}\n")

## Step 5: results
The fold-B table, the verdict line(s) and the list of output files.

In [ ]:
log = open(LOG, encoding="utf-8").read()
start = log.rfind("=== Fold B OOF")
print(log[start:start + 5000] if start >= 0 else log[-5000:])
print(subprocess.run(f"ls -la {OUT} {OUT}/variants", shell=True, capture_output=True, text=True).stdout)